<p align="center">
  <img src="archs/HyDE.webp" alt="HyDE">
</p>

In [ ]:
import re
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFacePipeline
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [3]:
loader = WebBaseLoader(web_path='https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', show_progress=True)
docs = loader.load()

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

In [9]:
# BGE models recommend normalized embeddings for optimal cosine similarity retrieval
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model='BAAI/bge-small-en-v1.5', encode_kwargs={"normalize_embeddings": True}, show_progress=True),
    collection_name="prompts",
    collection_metadata={"hnsw:space": "cosine"}
)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

In [10]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5, "lambda_mult": 0.5})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000020B66EA1A00>, search_type='mmr', search_kwargs={'k': 5, 'lambda_mult': 0.5})

In [45]:
llm_pipeline = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen3-0.6B",
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 2048,
        "temperature": 0.4,
        "do_sample": True,
        "return_full_text": False,
    }
)

llm = ChatHuggingFace(llm=llm_pipeline)

Device set to use cpu


In [33]:
def parse_hyde_output(response: str) -> str:
    """
    Extracts clean answer from Qwen3 output.
    Removes chat template tokens and thinking blocks.
    """
    # Remove chat template tokens only
    response = re.sub(r'<\|im_start\|>.*?\n', '', response)
    response = re.sub(r'<\|im_end\|>', '', response)
    
    # Remove thinking block — we let it think but don't include in hypothesis
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL)
    
    return response.strip()

In [42]:
prompt_hyde = PromptTemplate.from_template(""""
Write a short 2-3 sentence factual passage that directly answers: '{query}'
Write it as if it's an excerpt from a research paper or textbook.
Be specific and technical. No introduction, just the content.
""")

In [46]:
hyde_chain = prompt_hyde | llm | StrOutputParser() | RunnableLambda(parse_hyde_output)

In [47]:
print(hyde_chain.invoke("What is Prompt Engineering?"))

Prompt Engineering is a method that leverages natural language processing (NLP) and machine learning (ML) to design and refine prompts that achieve specific tasks. By structuring instructions and leveraging large-scale language models, it enables precise, efficient, and context-aware responses, optimizing performance and accuracy through targeted training and fine-tuning.


In [48]:
retriever.invoke("Prompt Engineering is a method that leverages natural language processing (NLP) and machine learning (ML) to design and refine prompts that achieve specific tasks. By structuring instructions and leveraging large-scale language models, it enables precise, efficient, and context-aware responses, optimizing performance and accuracy through targeted training and fine-tuning.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'language': 'en-us', 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heavy experimentation and heuristics.\nThis post only focuses on prompt engineering for autoregressive language models, so nothing with Cloze tests, image generation or multimodality models. At its core, the goal of prompt engineering is about alignment and model steerability. Check my previous post on controllable text generation.', 'title': "Prompt Engineering | Lil'Log"}, page_content='Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the m

In [50]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [62]:
prompt_rag = PromptTemplate.from_template("""
Use the provided context to answer the question.

If the context contains relevant information, answer using it.

If the context truly does not contain enough information, say:
"I don't know."

Context:
{context}

Question:
{question}

Answer:
""")

In [67]:
parallel_chain = RunnableParallel({
    'context': hyde_chain | retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

main_chain = parallel_chain | prompt_rag | llm | StrOutputParser() | RunnableLambda(parse_hyde_output)

In [68]:
print(main_chain.invoke("What is Chain of Thought Prompting?"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Chain of Thought Prompting is a method that generates a sequence of short sentences to describe reasoning logics step by step, helping to break down complex reasoning tasks into manageable steps.


In [69]:
print(main_chain.invoke("Who is Shinjan Saha and how is he related to Prompting?"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

I don't know.
